In [2]:
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import matplotlib.pyplot as plt
from pyspark.sql.types import *
from pyspark.sql import DataFrame, SparkSession
import seaborn as sns

In [3]:
spark = (
    SparkSession.builder
      .config("spark.driver.memory", "32g")                 # ajuste p/ 16g–24g
      .config("spark.sql.execution.arrow.pyspark.enabled", "true")
      .config("spark.sql.execution.arrow.maxRecordsPerBatch", "20000")
      .config("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024)  # 64MB/partição
      .config("spark.driver.maxResultSize", "0")            # sem limite de resultado (cuidado)
      .getOrCreate()
)

spark.version

'4.0.1'

In [4]:
# read table
#df = spark.read.option("mergeSchema", "true").parquet(
#    "/Volumes/sebraepe_dev/landing/raw/tests/testes_sample.parquet")

df = spark.read.option("header", True).parquet(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/dados/internacao.parquet"
)
df.count()

334560

In [5]:
# filter cases above august 2022
df = df.filter(F.col("dthr_internacao") >= F.lit("2022-08-01"))
df.count()


219622

In [6]:
# filtrar quem esta com data de internação em formato correto
df = df.filter(
    F.col("dthr_alta_medica").rlike(r'^\s*\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\s*$')
)
df = df.filter(
    F.col("dthr_internacao").rlike(r'^\s*\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\s*$')
)
df.count()

208032

In [7]:
# converte as strings para timestamp
df = df.withColumn("dthr_internacao", F.to_timestamp("dthr_internacao")) \
       .withColumn("dthr_alta_medica", F.to_timestamp("dthr_alta_medica"))

# calcula a diferenca em dias (alta - internação)
df = df.withColumn("dias_internado",
                   F.round(F.datediff("dthr_alta_medica", "dthr_internacao"), 2))

In [8]:
# targets
df = df.withColumn("mais_de_4_dias", F.when(F.col("dias_internado") > 4, 1).otherwise(0))

df = df.withColumn("mais_de_7_dias", F.when(F.col("dias_internado") > 7, 1).otherwise(0))

df = df.withColumn("mais_de_15_dias", F.when(F.col("dias_internado") > 15, 1).otherwise(0))

df.count()

208032

In [9]:
df = df.dropDuplicates(["prontuario", "dthr_internacao"])
df.count()

60847

In [10]:
# time to string
df = df.withColumn(
    "date_ref",
    F.date_trunc("month", F.col("dthr_alta_medica"))
)

In [11]:
df.select("prontuario").distinct().count()

24214

In [12]:
sample_df = df.orderBy(F.rand(seed=42)).limit(40000)

In [13]:
sample_df = sample_df.withColumn(
    "date_ref",
    F.date_format(F.col("date_ref"), "yyyy-MM-dd HH:mm:ss")
)

In [14]:
sample_df = sample_df.select("prontuario", 
                            "date_ref",
                            "dias_internado", 
                            "mais_de_4_dias", 
                            "mais_de_7_dias",
                            "mais_de_15_dias")

In [15]:
sample_df.select("prontuario").distinct().count()

17291

In [16]:
sample_df.printSchema()

root
 |-- prontuario: double (nullable = true)
 |-- date_ref: string (nullable = true)
 |-- dias_internado: integer (nullable = true)
 |-- mais_de_4_dias: integer (nullable = false)
 |-- mais_de_7_dias: integer (nullable = false)
 |-- mais_de_15_dias: integer (nullable = false)



In [17]:
sample_df.toPandas().to_parquet("C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/sample_target_internacao.parquet")